# 81장 · FastAPI로 구현하는 할 일 CRUD

사용자가 완료 버튼을 누른 직후, 화면은 어떤 응답을 받아야 체크 표시를 바꿀 수 있을까요?

FastAPI 경로로 할 일을 추가·수정·삭제합니다. 브라우저의 목록은 서버가 확정한 ID와 처리 결과를 받은 뒤 갱신합니다.

실행을 확인한 커널은 Python 3.12.15입니다. 예제에 사용한 외부 패키지는 fastapi 0.135.0, ipython 9.17.1, pydantic 2.14.1입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## POST와 GET으로 등록·조회

단일 사용자 실습의 저장소가 번호를 발급합니다. 생성 요청은 title만 보내고, 응답 data에는 id·title·completed가 전부 들어옵니다. 아래는 메모리 저장 경로의 축약 구현입니다. 원본에서 관리하는 오류·저장 상태 외피 중 이번 실습은 data에 집중합니다.


### 컬렉션 경로와 한 항목의 경로

<p>POST /todos는 새 항목을 만들고, GET /todos는 전체 목록을 읽습니다. PATCH와 DELETE는 /todos/1처럼 대상 ID를 경로에 담아 한 항목에 적용합니다.</p>


### 셀 01 · FastAPI CRUD 경로 등록

POST는 양끝을 다듬은 제목에 새 ID를 배정합니다. PATCH 예제는 완료 변경에 집중하고 DELETE는 삭제한 항목을 돌려줍니다.

노트북 축약판의 라우트부터 읽으세요. 참고 앱의 요청표에는 개별 조회와 제목 변경까지 있지만 축약 라우트에 없는 동작은 여기서 시험할 수 없습니다. 같은 PATCH라도 받는 필드가 completed인지 title인지 확인한 뒤 비교하세요.


In [1]:
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel, ConfigDict, StrictBool
from itertools import count

app = FastAPI()
items, ids = {}, count(1)
TRIM = "".join(chr(n) for n in (9, 10, 11, 12, 13, 32, 160, 0x1680, *range(0x2000, 0x200B), 0x2028, 0x2029, 0x202F, 0x205F, 0x3000, 0xFEFF))
class Create(BaseModel):
    model_config = ConfigDict(extra="forbid")
    title: str
class Complete(BaseModel):
    model_config = ConfigDict(extra="forbid")
    completed: StrictBool

def existing(id):
    if id not in items:
        raise HTTPException(404)
    return items[id]

@app.post("/todos", status_code=201)
def create(body: Create):
    title = body.title.strip(TRIM)
    if not title:
        raise HTTPException(422)
    key = next(ids)
    items[key] = {"id": key, "title": title, "completed": False}
    return {"data": items[key]}

@app.get("/todos")
def list_items():
    return {"data": list(items.values())}

@app.patch("/todos/{id}")
def patch(id: int, body: Complete):
    row = existing(id)
    row["completed"] = body.completed
    return {"data": row}

@app.delete("/todos/{id}")
def remove(id: int):
    existing(id)
    return {"data": items.pop(id)}

client = TestClient(app)

TestClient가 호출할 네 경로를 준비했습니다. 등록할 때는 제목의 양끝 공백을 없애고 새 ID를 발급합니다. 이 노트북의 PATCH는 completed만 받습니다. 아래 ZIP은 개별 조회와 제목 수정까지 구현한 강사용 참고 코드입니다. 수업 실습은 다음 셀에서 같은 TestClient로 계속합니다.


### 완료 입력에서 StrictBool 사용하기

<p>Complete 모델의 completed는 StrictBool입니다. JSON의 true와 문자열 "true"는 자료형이 다릅니다. 모델을 통과한 값만 경로 함수에 전달되므로 화면도 불리언을 보내야 합니다.</p>


<p><a download="chapter81_reference.zip" href="">강사용 참고 코드 · 개별 조회와 제목 수정 (ZIP)</a></p>


### 셀 02 · POST 응답에서 ID 읽기

POST 응답에서 서버가 정한 번호를 보관합니다.


In [2]:
created = client.post("/todos", json={"title": "  복습  "})
item = created.json()["data"]
print(created.status_code, item)

201 {'id': 1, 'title': '복습', 'completed': False}


201 응답에 담긴 id는 서버에서 발급한 값입니다. 다음 PATCH는 이 값을 경로에 넣습니다. 목록의 첫 줄을 수정 대상으로 삼으면 정렬 순서가 달라질 때 다른 항목을 고칠 수 있습니다.


#### 서버가 만든 할 일과 화면의 행 연결하기

<mark>후속 수정과 삭제에는 생성 응답이 돌려준 ID를 씁니다.</mark> 첫 줄에 보이는 행을 지운다고 해서 그 ID가 항상 1인 것은 아닙니다. 서버에서 같은 ID를 다시 조회해 수정된 완료값과 삭제 결과를 확인하세요.

선수 개념 · 68장 FastAPI 요청 모델과 POST 엔드포인트

## PATCH·DELETE 이후 화면 상태 갱신

완료 버튼을 눌렀을 때 기존 상태를 뒤집으라고 요청하면 두 번 전송했을 때 원래대로 돌아갈 수 있습니다. 원하는 최종 값 true를 보내고 반환된 항목을 화면에 반영합니다.


### 셀 03 · PATCH 후 목록 조회

반환 상태와 목록의 같은 ID가 일치하는지 확인합니다.


In [3]:
response = client.patch(f"/todos/{item['id']}", json={"completed": True})
listed = client.get("/todos").json()["data"][0]
print(response.status_code, listed["id"], listed["completed"])

200 1 True


200 1 True입니다. 제목이 같은 다른 항목이 있어도 요청 경로의 번호가 대상 행을 가리킵니다.


### 수정 응답 이후 목록에서 확인할 값

<p>이 예제는 POST를 한 번만 실행하므로 목록의 첫 행이 방금 만든 항목입니다. 여러 건을 등록했다면 응답 목록에서 item의 ID로 해당 행을 찾아야 합니다. 200, 1, True는 각각 요청 성공, 대상 ID, 현재 완료 상태를 뜻합니다.</p>


### 셀 04 · 응답 목록의 HTML 렌더링

앞에서 받은 GET의 data를 사용해 노트북 안에 완료 목록을 그립니다. 이어서 없는 항목의 실패 응답을 받아도 직전에 확인한 목록을 유지합니다.


In [4]:
from html import escape
from IPython.display import display

visible = client.get("/todos").json()["data"]
failed = client.patch("/todos/999", json={"completed": True})
if failed.is_success:
    changed = failed.json()["data"]
    visible = [changed if row["id"] == changed["id"] else row for row in visible]
labels = [f"{row['title']} · {'완료' if row['completed'] else '미완료'}" for row in visible]
html = "<ul>" + "".join("<li>" + escape(label) + "</li>" for label in labels) + "</ul>"
display({"text/html": html, "text/plain": " / ".join(labels)}, raw=True)
print(failed.status_code)

복습 · 완료

404


목록에는 ‘복습 · 완료’가 남고 실패한 요청의 상태는 404입니다. 목록은 실제 TestClient 응답으로 그렸습니다. 독립적인 브라우저가 원격 서버를 호출한 실험은 아닙니다.


### 404 뒤에 기존 목록을 보존하는 분기

<p>999번 항목에 PATCH를 보내면 404가 반환됩니다. 변경 결과로 visible을 교체하는 줄은 is_success일 때만 실행되므로 기존 1번 항목은 화면에 남습니다.</p>


### 과제 05 · DELETE 결과 검증

할 일을 삭제한 뒤 같은 ID로 한 번 더 삭제를 요청하세요. 두 응답의 상태 코드와 마지막 목록을 비교하면 재시도가 실제 저장 상태에 미치는 영향을 확인할 수 있습니다.


In [5]:
# 같은 ID에 DELETE를 두 번 보내고 GET /todos로 남은 목록을 읽으세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>path = f&quot;/todos/{item[&#x27;id&#x27;]}&quot;
removed = client.delete(path)
again = client.delete(path)
print(removed.status_code, again.status_code)
print(client.get(&quot;/todos&quot;).json()[&quot;data&quot;])</code></pre>
<p>실행 결과</p><pre><code>200 404
[]</code></pre>
<p>첫 요청에서 항목을 꺼냈으므로 같은 ID로 보낸 두 번째 요청에서는 existing이 404를 냅니다. 두 응답 코드는 달라도 최종 목록은 빈 배열입니다. 이 과제를 다시 실행할 때는 셀 01부터 시작해 항목을 새로 만드세요.</p>
</details>


#### DELETE 응답과 GET 목록이 확인하는 서로 다른 값

```mermaid
flowchart TD
nbefore["기존 항목"]
  ndelete["DELETE"]
  nresponse["응답 ID"]
  nlist["GET /todos"]
  nbefore --> ndelete
  ndelete --> nresponse
  nresponse --> nlist
```

삭제 응답에서 어떤 ID를 지웠는지 보고, 새 GET 요청으로 진짜 목록에서 사라졌는지 확인합니다.

## 제목 수정 API에 빈 문자열이 들어오면 어떻게 될까요?

기존 API는 완료 상태를 바꾸고 있습니다. 이번에는 제목 수정만 담당하는 작은 경로를 만듭니다. 등록한 할 일의 ID와 완료 값은 보존하고, 검증에 실패한 제목은 화면이나 저장값에 반영하지 않도록 합니다.

### 셀 07 · 수정 대상을 만들고 ID로 원래 상태 보관하기

셀 01에서 앱과 client를 정의하세요. 과제의 앞 셀에서 첫 항목을 이미 지웠어도 실습할 수 있도록 새 “초안” 한 건을 등록한 뒤 완료 처리합니다. FastAPI 응답에서 새 ID를 꺼내세요.

In [6]:
rename_seed = client.post("/todos", json={"title": "초안"}).json()["data"]
rename_id = rename_seed["id"]
before_rename = client.patch(f"/todos/{rename_id}", json={"completed": True}).json()["data"]
print(before_rename["title"], before_rename["completed"])

초안 True


초안 True는 앞으로 수정할 행의 현재 상태입니다. 새 제목을 반영한 뒤에도 같은 ID와 완료 값이 돌아와야 합니다.

### 과제 08 · 제목 필드만 바꾸는 경로 작성하기

기존 items와 existing을 사용하는 실습 앱에 PUT /todos/{todo_id}/title을 만드세요. Create 모델로 받은 제목을 등록 때처럼 정리하고, 공백뿐이면 422로 끝내세요. 정상 수정 뒤 공백을 다시 보내 결과를 확인하세요.

In [7]:
rename_app = FastAPI()
# 기존 existing, Create, TRIM을 재사용해 제목 수정 경로를 만드세요.
# 정상 제목과 공백 제목을 차례로 보내고 실제 저장된 행을 확인하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>rename_app = FastAPI()

@rename_app.put(&quot;/todos/{todo_id}/title&quot;)
def rename(todo_id: int, body: Create):
    row = existing(todo_id)
    title = body.title.strip(TRIM)
    if not title:
        raise HTTPException(422, &quot;제목을 입력하세요.&quot;)
    row[&quot;title&quot;] = title
    return {&quot;data&quot;: row}

with TestClient(rename_app) as rename_client:
    changed = rename_client.put(f&quot;/todos/{rename_id}/title&quot;, json={&quot;title&quot;: &quot;  발표 초안  &quot;})
    refused = rename_client.put(f&quot;/todos/{rename_id}/title&quot;, json={&quot;title&quot;: &quot;  &quot;})
stored = existing(rename_id)
print(changed.status_code, refused.status_code, stored[&quot;title&quot;], stored[&quot;completed&quot;])</code></pre>
<p>실행 결과</p><pre><code>200 422 발표 초안 True</code></pre>
<p>200 422 발표 초안 True가 나옵니다. 정상 요청이 제목을 정리해 저장했고, 두 번째 요청은 row를 고치기 전에 반환됐습니다. 화면은 422에서 기존 제목을 보존하고 78장에서 만든 오류 문구를 보여 줄 수 있습니다.</p>
</details>

<mark>필드 하나를 수정하는 응답에서도 건드리지 않은 필드의 값을 확인하세요.</mark> 완료 값을 False로 초기화하는 잘못된 구현에서도 제목만 출력하면 발표 초안이 정상으로 보입니다. 이 예제는 처음부터 완료된 행을 골라 그 차이를 드러냅니다.

메모리 저장소를 DB로 옮길 때도 UPDATE에서 바꿀 열을 명시하고 같은 요청 두 건으로 검산하세요.

## 돌아보기

PATCH에 JSON 문자열 "true"를 넣었을 때와 불리언 true를 넣었을 때를 비교해 보세요. StrictBool이 문자열을 거절하면 목록은 그대로여야 합니다. 응답 코드와 GET으로 읽은 행이 각각 어떤 사실을 알려주는지 구분합니다.


### 메모리 목록을 파일 DB로 이어가기

<p>이 장은 앱의 Python 딕셔너리에 할 일을 보관합니다. 다음 장에서는 SQLite 파일에 같은 ID·제목·완료 값을 저장하고 연결을 닫았다 다시 열어 값을 확인합니다.</p>
